# Setup

This notebook requires:
- [DeepFurniture Dataset](https://huggingface.co/datasets/byliu/DeepFurniture#using-the-dataset) downloaded and uncompressed to `./data/DeepFurniture/uncompressed_data`
- Repository [Depth-Anything-V2](https://github.com/DepthAnything/Depth-Anything-V2?tab=readme-ov-file#usage) cloned at `./DepthAnythingV2`
- Depth-Anything-V2 [pre-trained models](https://github.com/DepthAnything/Depth-Anything-V2?tab=readme-ov-file#pre-trained-models) downloaded to `./DepthAnythingV2/checkpoints`

In [1]:
import time
import random

import cv2
import numpy as np

import wandb

import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset, DataLoader, Subset

import torchvision.transforms as T
import torch.nn.functional as F

from tqdm import tqdm
from pathlib import Path

from DepthAnythingV2.depth_anything_v2.dpt import DepthAnythingV2

# reference to cloned repository
from data.DeepFurniture.deepfurniture import DeepFurnitureDataset


DEVICE = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

xFormers not available
xFormers not available


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.mps.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED) 

In [ ]:
wandb.login()

# Preprocessing & Dataset

In [ ]:
class DepthAnythingFurnitureDataset(Dataset):
    def __init__(self, data_list):
        self.data = data_list

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

In [ ]:
def collate_fn(batch):
    return [item for item in batch if item is not None]

In [ ]:
def preprocess_dataset(dataset, split, image_size=(392, 392)):
    processed_data = []

    transform_image = T.Compose([
        T.Resize(image_size),
        T.ToTensor(),
        T.Normalize(mean=[0.5]*3, std=[0.5]*3)
    ])

    transform_depth = T.Compose([
        T.Resize(image_size),
        T.ToTensor()
    ])

    for data in tqdm(dataset, desc="Preprocessing"):
        try:
            if not data['image'] or not data['depth']:
                continue

            image = transform_image(data['image'])
            depth = transform_depth(data['depth'])

            # Invert depth
            depth = 1.0 / (depth + 1e-6)

            processed_data.append({
                "image": image,
                "depth": depth
            })
        except Exception as e:
            print(f"Skipping corrupted sample: {e}")
            continue

    torch.save(processed_data, f"processed_{split}_dataset.pt")
    return processed_data

In [ ]:
base_dataset = DeepFurnitureDataset("./data/DeepFurniture/uncompressed_data")

# Split indices
num_samples = len(base_dataset)
indices = list(range(num_samples))
random.shuffle(indices)

train_split = int(0.8 * num_samples)
val_split = int(0.9 * num_samples)

train_indices = indices[:train_split]
val_indices = indices[train_split:val_split]
test_indices = indices[val_split:]

# Create subsets
train_subset = Subset(base_dataset, train_indices)
val_subset = Subset(base_dataset, val_indices)
test_subset = Subset(base_dataset, test_indices)

# Wrap subsets with image/depth transformations
train_dataset = DepthAnythingFurnitureDataset(preprocess_dataset(train_subset, 'train'))
val_dataset = DepthAnythingFurnitureDataset(preprocess_dataset(val_subset, 'val'))
test_dataset = DepthAnythingFurnitureDataset(preprocess_dataset(test_subset, 'test'))

# Model

In [ ]:
model_configs = {
    'vits': {'encoder': 'vits', 'features': 64, 'out_channels': [48, 96, 192, 384]},
    'vitb': {'encoder': 'vitb', 'features': 128, 'out_channels': [96, 192, 384, 768]},
    'vitl': {'encoder': 'vitl', 'features': 256, 'out_channels': [256, 512, 1024, 1024]},
    'vitg': {'encoder': 'vitg', 'features': 384, 'out_channels': [1536, 1536, 1536, 1536]}
}

encoder = 'vits' # or 'vits', 'vitb', 'vitg'

model = DepthAnythingV2(**model_configs[encoder])
model.load_state_dict(torch.load(f'DepthAnythingV2/checkpoints/depth_anything_v2_{encoder}.pth', map_location='cpu'))
model = model.to(DEVICE).eval()

# Training

In [ ]:
num_epochs = 5
run_name = f'depthanything-vits-{time.strftime("%Y%m%d-%H%M%S")}'

train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_fn
)
val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=collate_fn
)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
model.to(DEVICE)

wandb.init(project="hslu-dspro2-beyond2d", name=run_name, config={
    "epochs": num_epochs,
    "batch_size": train_loader.batch_size,
    "lr": optimizer.param_groups[0]["lr"],
    "encoder": encoder,
})

best_val_absrel = float("inf")
checkpoint_path = Path("best_model.pth")

for epoch in range(num_epochs):
    model.train()
    epoch_loss = 0.0
    total_absrel_train = 0.0
    total_delta1_train = 0.0
    start_time = time.time()

    print(f"\n🚀 Starting Epoch {epoch+1}/{num_epochs}")
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}", leave=False)

    for batch in progress_bar:
        images = torch.stack([b["image"] for b in batch]).to(DEVICE)
        depths = torch.stack([b["depth"].squeeze(0) for b in batch]).to(DEVICE)

        pred = model(images)
        loss = F.l1_loss(pred, depths)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

        absrel = (torch.abs(pred - depths) / depths.clamp(min=1e-6)).mean().item()
        total_absrel_train += absrel

        ratio = torch.max(pred / depths.clamp(min=1e-6), depths / pred.clamp(min=1e-6))
        delta1 = (ratio < 1.25).float().mean().item()
        total_delta1_train += delta1

        progress_bar.set_postfix(loss=loss.item(), absRel=absrel, delta1=delta1)

    avg_loss = epoch_loss / len(train_loader)
    avg_absrel_train = total_absrel_train / len(train_loader)
    avg_delta1_train = total_delta1_train / len(train_loader)
    elapsed = time.time() - start_time
    print(f"✅ Epoch {epoch+1} | Loss: {avg_loss:.4f} | absRel: {avg_absrel_train:.4f} | delta1: {avg_delta1_train:.4f} | Time: {elapsed:.2f}s")

    # Validation
    model.eval()
    val_loss = 0.0
    total_absrel_val = 0.0
    total_delta1_val = 0.0

    with torch.no_grad():
        val_bar = tqdm(val_loader, desc="Validation", leave=False)
        for batch in val_bar:
            images = torch.stack([b["image"] for b in batch]).to(DEVICE)
            depths = torch.stack([b["depth"].squeeze(0) for b in batch]).to(DEVICE)

            pred = model(images)
            loss = F.l1_loss(pred, depths)
            val_loss += loss.item()

            absrel = (torch.abs(pred - depths) / depths.clamp(min=1e-6)).mean().item()
            ratio = torch.max(pred / depths.clamp(min=1e-6), depths / pred.clamp(min=1e-6))
            delta1 = (ratio < 1.25).float().mean().item()

            total_absrel_val += absrel
            total_delta1_val += delta1

            val_bar.set_postfix(loss=loss.item(), absRel=absrel, delta1=delta1)

    avg_val_loss = val_loss / len(val_loader)
    avg_absrel_val = total_absrel_val / len(val_loader)
    avg_delta1_val = total_delta1_val / len(val_loader)
    print(f"🔍 Validation | Loss: {avg_val_loss:.4f} | absRel: {avg_absrel_val:.4f} | delta1: {avg_delta1_val:.4f}")

    wandb.log({
        "train_loss": avg_loss,
        "train_absRel": avg_absrel_train,
        "train_delta1": avg_delta1_train,
        "val_loss": avg_val_loss,
        "val_absRel": avg_absrel_val,
        "val_delta1": avg_delta1_val,
        "epoch": epoch + 1
    })

    if avg_absrel_val < best_val_absrel:
        best_val_absrel = avg_absrel_val
        torch.save(model.state_dict(), checkpoint_path)
        print(f"💾 Best model saved with absRel={avg_absrel_val:.4f}")


wandb.finish()


# Evaluation

In [ ]:
import gc

del model, train_loader, val_loader, optimizer

gc.collect()

torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [ ]:
pretrained_model = DepthAnythingV2(**model_configs[encoder])
pretrained_model.load_state_dict(torch.load(f'DepthAnythingV2/checkpoints/depth_anything_v2_{encoder}.pth', map_location='cpu'))
pretrained_model = pretrained_model.to(DEVICE).eval()

In [ ]:
finetuned_model = DepthAnythingV2(**model_configs[encoder])
finetuned_model.load_state_dict(torch.load(f'best_model.pth', map_location='cpu'))
finetuned_model = finetuned_model.to(DEVICE).eval()

In [ ]:
def compute_depth_anything(model, image):
    img_rgb = np.array(image)
    
    # Convert to BGR format for OpenCV
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    
    with torch.no_grad():
        depth_map = model.infer_image(img_bgr)
    
    return depth_map

In [ ]:
def convert_tensor_image(image):
    # Convert torch.Tensor to NumPy array if needed
    if isinstance(image, torch.Tensor):
        image = image.permute(1, 2, 0).cpu().numpy()  # CxHxW → HxWxC
        image = (image * 255).astype(np.uint8)  # Rescale if image was normalized

    # Convert to BGR format for OpenCV
    return cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

In [ ]:
def compute_depth_anything(model, image):
    img_bgr = convert_tensor_image(image)

    with torch.no_grad():
        depth_map = model.infer_image(img_bgr)

    return depth_map

In [ ]:
test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
image = None
depth = None

for scene in test_loader:
    image = scene[0]['image']  # Assuming batch size is 1
    depth = scene[0]['depth']

    output_pretrained = compute_depth_anything(pretrained_model, image)
    output_finetuned = compute_depth_anything(finetuned_model, image)
    break  # Remove this line if you want to run through the full test set

In [ ]:
image = convert_tensor_image(image)
depth = convert_tensor_image(depth)

fig, axes = plt.subplots(1, 4, figsize=(16, 8))

axes[0].set_title('RGB Image', fontsize=14)
axes[0].imshow(image)

axes[1].set_title('Dataset Depth', fontsize=14)
axes[1].imshow(depth)

axes[2].set_title('DepthAnythingV2 Pretrained', fontsize=14)
axes[2].imshow(output_pretrained)

axes[3].set_title('DepthAnythingV2 Finetuned', fontsize=14)
axes[3].imshow(output_finetuned)